# XAI-ED — Notebook 4: End-to-End Student Case Studies

This notebook walks through **four representative student profiles** — one per risk tier —
demonstrating the full XAI-ED pipeline from prediction to intervention:

| Case | Profile | Risk Tier |
|------|---------|----------|
| A | Strong performer | **Strong** |
| B | Mild risk | **Borderline** |
| C | Moderate at-risk, actionable | **At Risk** |
| D | High risk, structural barriers | **High Risk** |

Each case includes:
- Feature profile
- Mastery prediction and risk tier
- SHAP local explanation
- Counterfactual recommendation
- Student-friendly explanation
- Instructor report excerpt

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.config import FEATURE_COLUMNS, TARGET_COLUMN
from src.data_loader import load_dataset
from src.train_model import train
from src.explain import _extract_class1_shap
from src.counterfactual import generate_counterfactual
from src.translator import summarize_shap_to_text, summarize_for_instructor, _risk_tier
from sklearn.model_selection import train_test_split

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

FRIENDLY = {
    'study_time_min': 'Study Time (min/wk)',
    'practice_completion_rate': 'Practice Completion',
    'avg_quiz_score': 'Avg Quiz Score',
    'quiz_attempts': 'Quiz Attempts',
    'hint_usage_rate': 'Hint Usage Rate',
    'attendance_rate': 'Attendance Rate',
    'days_since_last_activity': 'Days Since Last Activity',
    'stress_index': 'Stress Index',
    'prereq_mastery': 'Prereq Mastery',
    'device_reliability': 'Device Reliability',
}

print('Libraries loaded.')

In [ ]:
X, y = load_dataset('../data/student_data.csv')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training RF model...')
trained_rf  = train('rf', X_train, y_train)
rf_pipeline = trained_rf.pipeline
preprocess  = rf_pipeline.named_steps['preprocess']
clf         = rf_pipeline.named_steps['clf']
explainer   = shap.TreeExplainer(clf)
print('Model and explainer ready.')

In [ ]:
def explain_student(row: pd.Series, model_name='rf'):
    """Full XAI-ED pipeline for one student row."""
    prob = float(rf_pipeline.predict_proba(pd.DataFrame([row]))[:, 1][0])
    tier = _risk_tier(prob)

    Xe      = preprocess.transform(pd.DataFrame([row]))
    sv_raw  = explainer.shap_values(Xe)
    sv      = _extract_class1_shap(sv_raw)[0]

    cf = generate_counterfactual(rf_pipeline, row.copy())

    student_text = summarize_shap_to_text(
        FEATURE_COLUMNS, row.to_dict(), sv.tolist(), prob
    )
    instructor_text = summarize_for_instructor(
        FEATURE_COLUMNS, row.to_dict(), sv.tolist(), prob,
        model_name=model_name, counterfactual=cf
    )

    return dict(prob=prob, tier=tier, shap=sv, cf=cf,
                student_text=student_text,
                instructor_text=instructor_text)


def plot_local_shap(ax, sv, title, prob, tier):
    order  = np.argsort(np.abs(sv))[::-1]
    top    = order[:8]
    vals   = sv[top]
    labels = [FRIENDLY.get(FEATURE_COLUMNS[i], FEATURE_COLUMNS[i]) for i in top]
    colors = ['#42a5f5' if v > 0 else '#ef5350' for v in vals]
    ax.barh(labels[::-1], vals[::-1], color=colors[::-1], edgecolor='white')
    ax.axvline(0, color='white', lw=0.8)
    ax.set_xlabel('SHAP value')
    ax.set_title(f'{title}\np={prob:.3f} ({tier})', fontweight='bold')


print('Helper functions defined.')

In [ ]:
# Find representative students for each tier
proba_all = rf_pipeline.predict_proba(X_test)[:, 1]
tiers_all = [_risk_tier(p) for p in proba_all]

def find_case(target_tier, exclude=set()):
    for i, t in enumerate(tiers_all):
        if t == target_tier and i not in exclude:
            return i
    return None

used = set()
case_a_idx = find_case('Strong',     used);  used.add(case_a_idx)
case_b_idx = find_case('Borderline', used);  used.add(case_b_idx)
case_c_idx = find_case('At Risk',    used);  used.add(case_c_idx)
case_d_idx = find_case('High Risk',  used);  used.add(case_d_idx)

print(f'Case A (Strong):     Student #{case_a_idx} — p={proba_all[case_a_idx]:.3f}')
print(f'Case B (Borderline): Student #{case_b_idx} — p={proba_all[case_b_idx]:.3f}')
print(f'Case C (At Risk):    Student #{case_c_idx} — p={proba_all[case_c_idx]:.3f}')
print(f'Case D (High Risk):  Student #{case_d_idx} — p={proba_all[case_d_idx]:.3f}')

## Case Studies: SHAP Explanations (All 4 Tiers)

In [ ]:
cases = [
    ('A — Strong',     case_a_idx),
    ('B — Borderline', case_b_idx),
    ('C — At Risk',    case_c_idx),
    ('D — High Risk',  case_d_idx),
]
cases = [(label, idx) for label, idx in cases if idx is not None]

results = {}
for label, idx in cases:
    row     = X_test.iloc[idx]
    results[label] = explain_student(row)
    print(f'{label}: computed.')

fig, axes = plt.subplots(1, len(cases), figsize=(5 * len(cases), 6))
if len(cases) == 1:
    axes = [axes]

for ax, (label, idx) in zip(axes, cases):
    r = results[label]
    plot_local_shap(ax, r['shap'], f'Case {label}', r['prob'], r['tier'])

plt.suptitle('Local SHAP Explanations — Four Risk Tiers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/case_studies_shap.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Profiles

In [ ]:
for label, idx in cases:
    if idx is None:
        continue
    row  = X_test.iloc[idx]
    r    = results[label]
    print(f'\n{'='*60}')
    print(f'CASE {label}  |  p={r["prob"]:.3f}  |  Tier: {r["tier"]}')
    print('='*60)
    for feat in FEATURE_COLUMNS:
        print(f'  {FRIENDLY.get(feat, feat):<35} {row[feat]:.3f}')
    print(f'\nCounterfactual status: {r["cf"]["status"]}')
    if r['cf']['edits']:
        print('  Required changes:')
        for f, v in r['cf']['edits'].items():
            print(f'    {FRIENDLY.get(f, f)}: {row[f]:.3f} -> {v:.3f}')
    print(f'\nStudent message:\n  {r["student_text"]}')

## Instructor Reports

In [ ]:
for label, idx in cases:
    if idx is None:
        continue
    r = results[label]
    print(f'\n{"="*60}')
    print(f'CASE {label}')
    print('='*60)
    print(r['instructor_text'])

## Radar Chart: Feature Profiles Across Cases

In [ ]:
# Normalize features to [0,1] for radar
from src.config import FEATURE_COLUMNS
import matplotlib.patches as mpatches

radar_features = [
    'practice_completion_rate', 'attendance_rate', 'avg_quiz_score',
    'prereq_mastery', 'study_time_min', 'hint_usage_rate',
    'days_since_last_activity', 'stress_index'
]
radar_labels = [FRIENDLY.get(f, f) for f in radar_features]

# Normalize to [0,1]
X_min = X_train[radar_features].min()
X_max = X_train[radar_features].max()

N = len(radar_features)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(10, 10))
ax  = fig.add_subplot(111, polar=True)
colors_cases = ['#42a5f5', '#ffa726', '#ef5350', '#ab47bc']

for (label, idx), color in zip(cases, colors_cases):
    if idx is None:
        continue
    row = X_test.iloc[idx]
    vals_norm = [(row[f] - X_min[f]) / max(X_max[f] - X_min[f], 1e-9) for f in radar_features]
    vals_norm += vals_norm[:1]
    ax.plot(angles, vals_norm, label=f'Case {label}', color=color, lw=2)
    ax.fill(angles, vals_norm, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=9)
ax.set_ylim(0, 1)
ax.set_title('Student Feature Profiles (normalised)', size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('../outputs/case_studies_radar.png', dpi=150, bbox_inches='tight')
plt.show()

---
**Notebook series complete.**

All four notebooks together constitute a fully reproducible research demonstration of the XAI-ED system:

1. `01_data_exploration.ipynb` — Dataset, distributions, equity gaps
2. `02_model_comparison.ipynb` — Training, evaluation, statistical tests
3. `03_xai_deep_dive.ipynb` — SHAP, LIME, counterfactuals, fairness
4. `04_case_studies.ipynb` — End-to-end student case studies

For the interactive dashboard, run: `streamlit run dashboard.py`